Step 1: Bootstrap & install

In [ ]:
!pip install --quiet anthropic pydantic
from google.colab import userdata, drive # type: ignore
drive.mount('/content/drive')

import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["ASTRA_CASSETTE_DIR"] = "/content/drive/MyDrive/astra-swarm/cassettes"

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")

from astra_swarm import attack_kb
from astra_swarm.agent_loop import run_with_tools
from astra_swarm.cassette import cassette
from astra_swarm.tools import lookup_attack_technique_by_id, search_attack_techniques
from astra_swarm.alerts import (
    triage_chain, generate_synthetic_alerts,
    parse_alert, summarize, assess_severity,
)

print("ready")

Step 2: ID lookup from KB

In [ ]:
# Direct ID lookup — triggers the download on first call
t = lookup_attack_technique_by_id("T1078")
print(t["name"], "→ tactics:", t["tactics"])
print(t["description"][:200], "...")           # Update to get 
print()

# Sub-technique
t = lookup_attack_technique_by_id("T1078.004")
print(t["name"], "→ tactics:", t["tactics"])
print()

# Unknown ID
t = lookup_attack_technique_by_id("T9999")
print(t)
print()

# Keyword search
r = search_attack_techniques(keyword="mfa fatigue")
for m in r.get("matches", []):
    print(f"  {m['id']:<10} {m['name']}")

Step 3: Verify tool round trip

In [ ]:
with cassette("day04_tool_round_trip", mode="auto"):
    result = run_with_tools(
        "Explain in one paragraph what would motivate an adversary to combine "
        "MITRE ATT&CK T1078 (Valid Accounts) and T1621 (MFA request generation). "
        "Use the lookup tool to ground each technique. Cite tactics explicitly.",
        verbose=True,
        max_rounds=6,
    )
print()
print(result["final_text"])
print(f"\nrounds: {result['rounds']}, stop_reason: {result['stop_reason']}")

In [ ]:
# with a keyword call

with cassette("day04_keyword_call", mode="auto"):
    result = run_with_tools(
        "A user account received 47 push notifications in 5 minutes. "
        "Look up the relevant MITRE technique and explain in one sentence what's happening.",
        verbose=True,
        max_rounds=8,
    )
print(result["final_text"])

Step 4: Rerun full triage chain with enrichment

In [ ]:
import hashlib
import json
from pathlib import Path

try:
    alerts = json.loads(Path("/content/astra-swarm/data/synthetic/02_alerts.json").read_text())
    results = []
    for i, a in enumerate(alerts, 1):
        print(f"--- Alert {i} ---")
        alert_id = hashlib.md5(a.encode()).hexdigest()[:8]
        with cassette(f"day04_v1_enriched_alert{i:02d}_{alert_id}", mode="auto"):
            r = triage_chain(a)
            results.append(r)
            techs = r.attack.techniques
            print(f"  Cited ATT&CK: {', '.join(t.id for t in techs) or '(none)'}")
            print(f"  Severity: {r.verdict.severity} "
                f"(conf {r.verdict.confidence:.2f})")
            print()
except FileNotFoundError:
    print("Alert file not found.")

Step 5: Unmount & cleanup

In [ ]:
drive.flush_and_unmount()

Debug Cell (what Claude returned for failing alert)

In [ ]:
# Debug cell

# Cell — see exactly what Claude returned for the failing alert
from astra_swarm.alerts import parse_alert
from astra_swarm.agent_loop import run_with_tools
import json

raw = ("<timestamp>2024-01-15T09:47:23Z</timestamp> host=FIREWALL-01 severity=medium "
       "action=blocked src=192.168.45.67 dst=203.0.113.42 dport=445 protocol=tcp "
       "rule_id=FW-SMB-BLOCK threat_type=lateral_movement_attempt user=domain\\jsmith")

parsed = parse_alert(raw)
print("PARSED (this works):", json.dumps(parsed, indent=2))

prompt = f"""[same enrich_with_attack prompt as before]
Parsed alert:
{json.dumps(parsed, indent=2)}
"""
result = run_with_tools(prompt, max_rounds=6, max_tokens=1500)
print("\nRAW FINAL_TEXT (this is what fails to parse):")
print(repr(result["final_text"]))

Temporary Diagnostic Cell

In [ ]:
# Cell — isolate which chain step fails
import json, traceback
from astra_swarm.alerts import (
    parse_alert, summarize, assess_severity, enrich_with_attack, _parse_json,
)
from astra_swarm.agent_loop import run_with_tools

raw = ("<timestamp>2024-01-15T09:47:23Z</timestamp> host=FIREWALL-01 severity=medium "
       "action=blocked src=192.168.45.67 dst=203.0.113.42 dport=445 protocol=tcp "
       "rule_id=FW-SMB-BLOCK threat_type=lateral_movement_attempt user=domain\\jsmith")

# Sanity check: which _parse_json is loaded?
import inspect
src = inspect.getsource(_parse_json)
print(f"_parse_json length: {len(src)} chars — "
      f"{'looks like OLD version' if len(src) < 300 else 'looks like UPDATED version'}")
print("---")

for step_name, step_fn in [
    ("parse_alert", lambda: parse_alert(raw)),
]:
    try:
        parsed = step_fn()
        print(f"✓ {step_name} OK")
        print("parsed:", parsed)
        print(json.dumps(parsed, indent=2)[:400])
    except Exception as e:
        print(f"✗ {step_name} FAILED: {type(e).__name__}: {e}")
        traceback.print_exc()
        raise SystemExit

print("---")
try:
    enrichment = enrich_with_attack(parsed)
    print(f"✓ enrich_with_attack OK: {enrichment}")
except Exception as e:
    print(f"✗ enrich_with_attack FAILED: {type(e).__name__}: {e}")
    # Re-run just the LLM call so we can see the raw text that failed to parse
    prompt = f"[same prompt as enrich]\nParsed alert:\n{json.dumps(parsed, indent=2)}"
    result = run_with_tools(prompt, max_rounds=6, max_tokens=1500)
    print("\n=== RAW final_text that _parse_json choked on ===")
    print(repr(result["final_text"])[:2000])
    print(f"\nstop_reason: {result['stop_reason']}, rounds: {result['rounds']}")
    raise SystemExit

try:
    summary = summarize(parsed)
    print(f"✓ summarize OK")
except Exception as e:
    print(f"✗ summarize FAILED: {type(e).__name__}: {e}")
    raise SystemExit

try:
    verdict = assess_severity(parsed, summary)
    print(f"✓ assess_severity OK: {verdict}")
except Exception as e:
    print(f"✗ assess_severity FAILED: {type(e).__name__}: {e}")